In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.data.cleaning import load_and_clean_data
from src.data.aggregation import aggregate_to_order_level
from src.data.splitting import time_based_split

from src.features.preprocessing import (
    add_time_features,
    learn_rare_countries,
    group_rare_countries,
    build_preprocessor,
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    TARGET,
)

from src.models.evaluation import evaluate_classifier

In [3]:
df_clean = load_and_clean_data()

order_df = aggregate_to_order_level(df_clean)

train_df, val_df, test_df = time_based_split(order_df)

In [4]:
train_df = add_time_features(train_df)
val_df = add_time_features(val_df)
test_df = add_time_features(test_df)

rare_countries = learn_rare_countries(train_df)

train_df = group_rare_countries(train_df, rare_countries)
val_df = group_rare_countries(val_df, rare_countries)
test_df = group_rare_countries(test_df, rare_countries)

In [5]:
FEATURE_COLUMNS = CATEGORICAL_FEATURES + NUMERIC_FEATURES

X_train = train_df[FEATURE_COLUMNS]
X_val = val_df[FEATURE_COLUMNS]
X_test = test_df[FEATURE_COLUMNS]

y_train = train_df[TARGET]
y_val = val_df[TARGET]
y_test = test_df[TARGET]

preprocessor = build_preprocessor()

X_train_prepared = preprocessor.fit_transform(X_train)
X_val_prepared = preprocessor.transform(X_val)
X_test_prepared = preprocessor.transform(X_test)

In [6]:
print("X_train:", X_train_prepared.shape)
print("X_val:", X_val_prepared.shape)
print("X_test:", X_test_prepared.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (46026, 168)
X_val: (9863, 168)
X_test: (9863, 168)
y_train: (46026,)
y_val: (9863,)
y_test: (9863,)


In [10]:
from src.models.baseline import (
    build_baseline_model,
    train_baseline_model
)

baseline_model = build_baseline_model()
baseline_model = train_baseline_model(
    baseline_model,
    X_train_prepared,
    y_train
)

baseline_metrics = evaluate_classifier(
    baseline_model,
    X_val_prepared,
    y_val
)
baseline_metrics

{'accuracy': 0.7008009733346852,
 'precision': 0.8362930077691454,
 'recall': 0.5607441860465117,
 'f1': 0.6713442476890522,
 'roc_auc': 0.7429475604195166,
 'confusion_matrix': array([[3898,  590],
        [2361, 3014]])}

## 1- Decision Tree

In [13]:
from sklearn.tree import DecisionTreeClassifier

decision_tree = DecisionTreeClassifier(
    random_state=42
)

decision_tree.fit(
    X_train_prepared,
    y_train
)

decision_tree_metrics = evaluate_classifier(
    decision_tree,
    X_val_prepared,
    y_val
)

decision_tree_metrics

{'accuracy': 0.6343911588766096,
 'precision': 0.6626816258966342,
 'recall': 0.6703255813953488,
 'f1': 0.6664816870144284,
 'roc_auc': 0.6308401525515068,
 'confusion_matrix': array([[2654, 1834],
        [1772, 3603]])}

In [14]:
decision_tree_train_metrics = evaluate_classifier(
    decision_tree,
    X_train_prepared,
    y_train
)

decision_tree_train_metrics

{'accuracy': 1.0,
 'precision': 1.0,
 'recall': 1.0,
 'f1': 1.0,
 'roc_auc': 1.0,
 'confusion_matrix': array([[20783,     0],
        [    0, 25243]])}

## 2-Random Forest

In [18]:
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=1
)

random_forest.fit(
    X_train_prepared,
    y_train
)

rensom_forest_metrics = evaluate_classifier(
    random_forest,
    X_val_prepared,
    y_val
)

rensom_forest_metrics

{'accuracy': 0.7038426442258947,
 'precision': 0.7904829545454546,
 'recall': 0.6212093023255814,
 'f1': 0.6956974684863007,
 'roc_auc': 0.7615809186253782,
 'confusion_matrix': array([[3603,  885],
        [2036, 3339]])}

In [19]:
random_forest_train_metrics = evaluate_classifier(
    random_forest,
    X_train_prepared,
    y_train,
)

random_forest_train_metrics

{'accuracy': 1.0,
 'precision': 1.0,
 'recall': 1.0,
 'f1': 1.0,
 'roc_auc': 1.0,
 'confusion_matrix': array([[20783,     0],
        [    0, 25243]])}

## 3- XGBoost

In [20]:
%pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 7.1 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [21]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
)

xgb_model.fit(
    X_train_prepared,
    y_train,
)

xgb_metrics = evaluate_classifier(
    xgb_model,
    X_val_prepared,
    y_val,
)

xgb_metrics

{'accuracy': 0.7100273750380209,
 'precision': 0.8064830611747502,
 'recall': 0.6156279069767442,
 'f1': 0.6982485756488711,
 'roc_auc': 0.763946772789454,
 'confusion_matrix': array([[3694,  794],
        [2066, 3309]])}

In [22]:
xgb_train_metrics = evaluate_classifier(
    xgb_model,
    X_train_prepared,
    y_train,
)

xgb_train_metrics

{'accuracy': 0.7596358579933081,
 'precision': 0.8665219189412737,
 'recall': 0.6640256704829062,
 'f1': 0.75187835019176,
 'roc_auc': 0.8660418394753304,
 'confusion_matrix': array([[18201,  2582],
        [ 8481, 16762]])}

# paramerer tuning

## 1- Random Forest
max_depth
min_samples_split
min_samples_leaf
max_features
n_estimators

In [30]:
from sklearn.model_selection import ParameterGrid
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [10, 20, None],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt"]
}

results = []

for params in ParameterGrid(param_grid):
    model = RandomForestClassifier(
        **params,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train_prepared, y_train)
    metrics = evaluate_classifier(
        model,
        X_val_prepared,
        y_val
    )

    results.append({
        **params,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "roc_auc": metrics["roc_auc"],
    })

import pandas as pd

rf_tuning_results = pd.DataFrame(results)

rf_tuning_results = rf_tuning_results.sort_values(
    "roc_auc",
    ascending=False,
)

rf_tuning_results.head(10)

,max_depth,max_features,min_samples_leaf,n_estimators,accuracy,precision,recall,f1,roc_auc
8,20.0,sqrt,5,200,0.719152,0.848355,0.590140,0.696072,0.769695
9,20.0,sqrt,5,300,0.719152,0.848355,0.590140,0.696072,0.769566
11,20.0,sqrt,10,300,0.719152,0.848542,0.589953,0.696005,0.768818
10,20.0,sqrt,10,200,0.719152,0.848542,0.589953,0.696005,0.768030
17,NaN,sqrt,10,300,0.719254,0.848582,0.590140,0.696148,0.767641
15,NaN,sqrt,5,300,0.718848,0.846195,0.591628,0.696376,0.767607
16,NaN,sqrt,10,200,0.719254,0.848582,0.590140,0.696148,0.767564
14,NaN,sqrt,5,200,0.718747,0.845602,0.592000,0.696432,0.767500
6,20.0,sqrt,1,200,0.719254,0.848210,0.590512,0.696282,0.766558
0,10.0,sqrt,1,200,0.719152,0.848542,0.589953,0.696005,0.766074


In [31]:
best_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)

best_rf.fit(
    X_train_prepared,
    y_train,
)

best_rf_train_metrics = evaluate_classifier(
    best_rf,
    X_train_prepared,
    y_train,
)

best_rf_val_metrics = evaluate_classifier(
    best_rf,
    X_val_prepared,
    y_val,
)

print("Train:")
print(best_rf_train_metrics)

print("\nValidation:")
print(best_rf_val_metrics)

Train:
{'accuracy': 0.7182896623647503, 'precision': 0.8582015521970007, 'recall': 0.5826169631184883, 'f1': 0.6940537989617744, 'roc_auc': 0.8558671256073256, 'confusion_matrix': array([[18353,  2430],
       [10536, 14707]])}

Validation:
{'accuracy': 0.7191523877116496, 'precision': 0.8483551751805296, 'recall': 0.590139534883721, 'f1': 0.696071977177968, 'roc_auc': 0.769694565352568, 'confusion_matrix': array([[3921,  567],
       [2203, 3172]])}
